# Dataset Validation — BraTS 2021

## 1. Load Libraries

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
from collections import Counter

## 2. Load Case List

In [2]:
REPO_ROOT    = Path().resolve().parent
DATASET_ROOT = REPO_ROOT / 'data' / 'raw' / 'BraTS2021' / 'BraTS2021_Training_Data'

SUFFIXES = {
    't1':    '_t1.nii.gz',
    't1ce':  '_t1ce.nii.gz',
    't2':    '_t2.nii.gz',
    'flair': '_flair.nii.gz',
    'seg':   '_seg.nii.gz',
}

MODALITIES = ['t1', 't1ce', 't2', 'flair']
EXPECTED_SEG_LABELS = {0, 1, 2, 4}

cases = sorted([p for p in DATASET_ROOT.iterdir() if p.is_dir()])

print(f'Dataset root : {DATASET_ROOT}')
print(f'Total cases  : {len(cases)}')
print(f'First 5      : {[c.name for c in cases[:5]]}')
print(f'Last 5       : {[c.name for c in cases[-5:]]}')

Dataset root : C:\3D-Glioblastoma-MRI\data\raw\BraTS2021\BraTS2021_Training_Data
Total cases  : 1251
First 5      : ['BraTS2021_00000', 'BraTS2021_00002', 'BraTS2021_00003', 'BraTS2021_00005', 'BraTS2021_00006']
Last 5       : ['BraTS2021_01662', 'BraTS2021_01663', 'BraTS2021_01664', 'BraTS2021_01665', 'BraTS2021_01666']


## 3. Validate Required Files

In [3]:
file_check = []
for case in cases:
    cid = case.name
    present = {f.name for f in case.iterdir() if f.is_file()}
    row = {'case_id': cid}
    for mod, suffix in SUFFIXES.items():
        row[mod] = (cid + suffix) in present
    file_check.append(row)

df_files = pd.DataFrame(file_check).set_index('case_id')

print(f'Files present per modality:')
print(df_files.sum())
print(f'\nIncomplete cases: {(~df_files.all(axis=1)).sum()}')

Files present per modality:
t1       1251
t1ce     1251
t2       1251
flair    1251
seg      1251
dtype: int64

Incomplete cases: 0


## 4–8. Full Validation (one pass per case)

Check NIfTI loading, shapes, spacing, segmentation labels, and NaN/Inf.

In [4]:
results = []

for i, case in enumerate(cases):
    cid = case.name
    row = {
        'case_id':             cid,
        'files_complete':      bool(df_files.loc[cid].all()),
        'nifti_readable':      True,
        'shape_consistent':    True,
        'spacing_consistent':  True,
        'segmentation_valid':  True,
        'has_nan_or_inf':      False,
        'shape':               None,
        'spacing':             None,
        'seg_labels':          None,
        'unreadable_files':    [],
        'nan_inf_files':       [],
    }

    shapes   = {}
    spacings = {}

    # --- load all 5 volumes ---
    for mod, suffix in SUFFIXES.items():
        fpath = case / (cid + suffix)
        if not fpath.exists():
            continue
        try:
            img  = nib.load(str(fpath))
            data = np.asarray(img.dataobj)
        except Exception:
            row['nifti_readable'] = False
            row['unreadable_files'].append(mod)
            continue

        shapes[mod]   = img.shape
        spacings[mod] = tuple(round(float(v), 4) for v in img.header.get_zooms())

        # segmentation label check
        if mod == 'seg':
            labels = set(np.unique(data).tolist())
            row['seg_labels'] = sorted(labels)
            if not labels.issubset(EXPECTED_SEG_LABELS):
                row['segmentation_valid'] = False

        # NaN / Inf check (MRI volumes only)
        if mod in MODALITIES:
            if np.any(np.isnan(data)) or np.any(np.isinf(data)):
                row['has_nan_or_inf'] = True
                row['nan_inf_files'].append(mod)

    # shape consistency
    unique_shapes = set(shapes.values())
    if len(unique_shapes) != 1:
        row['shape_consistent'] = False
    row['shape'] = list(unique_shapes)

    # spacing consistency
    unique_spacings = set(spacings.values())
    if len(unique_spacings) != 1:
        row['spacing_consistent'] = False
    row['spacing'] = list(unique_spacings)

    results.append(row)

    if (i + 1) % 100 == 0 or (i + 1) == len(cases):
        print(f'  Validated {i + 1}/{len(cases)} cases')

df_val = pd.DataFrame(results).set_index('case_id')
print('\nValidation complete.')

  Validated 100/1251 cases
  Validated 200/1251 cases
  Validated 300/1251 cases
  Validated 400/1251 cases
  Validated 500/1251 cases
  Validated 600/1251 cases
  Validated 700/1251 cases
  Validated 800/1251 cases
  Validated 900/1251 cases
  Validated 1000/1251 cases
  Validated 1100/1251 cases
  Validated 1200/1251 cases
  Validated 1251/1251 cases

Validation complete.


## 5. Shape Distribution

In [5]:
shape_counts = Counter()
for shapes_list in df_val['shape']:
    for s in shapes_list:
        shape_counts[s] += 1

print('Shape distribution across cases:')
for shape, count in shape_counts.most_common():
    print(f'  {shape} : {count} cases')

mismatch = df_val[~df_val['shape_consistent']]
print(f'\nCases with shape mismatch: {len(mismatch)}')
if len(mismatch) > 0:
    print(mismatch[['shape']])

Shape distribution across cases:
  (240, 240, 155) : 1251 cases

Cases with shape mismatch: 0


## 6. Voxel Spacing Distribution

In [6]:
spacing_counts = Counter()
for spacings_list in df_val['spacing']:
    for s in spacings_list:
        spacing_counts[s] += 1

print('Voxel spacing distribution:')
for spacing, count in spacing_counts.most_common():
    print(f'  {spacing} : {count} cases')

spacing_mismatch = df_val[~df_val['spacing_consistent']]
print(f'\nCases with spacing mismatch across modalities: {len(spacing_mismatch)}')
if len(spacing_mismatch) > 0:
    print(spacing_mismatch[['spacing']])

Voxel spacing distribution:
  (1.0, 1.0, 1.0) : 1251 cases

Cases with spacing mismatch across modalities: 0


## 7. Segmentation Labels

In [7]:
label_counts = Counter()
for labels in df_val['seg_labels']:
    if labels is not None:
        label_counts[tuple(labels)] += 1

print('Segmentation label sets found:')
for label_set, count in label_counts.most_common():
    print(f'  {list(label_set)} : {count} cases')

invalid_seg = df_val[~df_val['segmentation_valid']]
print(f'\nCases with unexpected labels: {len(invalid_seg)}')
if len(invalid_seg) > 0:
    print(invalid_seg[['seg_labels']])

Segmentation label sets found:
  [0, 1, 2, 4] : 1180 cases
  [0, 2, 4] : 37 cases
  [0, 1, 2] : 27 cases
  [0, 2] : 6 cases
  [0, 1, 4] : 1 cases

Cases with unexpected labels: 0


## 8. NaN / Inf Check

In [8]:
nan_inf_cases = df_val[df_val['has_nan_or_inf']]
print(f'Cases with NaN or Inf values: {len(nan_inf_cases)}')
if len(nan_inf_cases) > 0:
    print(nan_inf_cases[['nan_inf_files']])

Cases with NaN or Inf values: 0


## 9. Validation Summary

In [9]:
summary_cols = ['files_complete', 'nifti_readable', 'shape_consistent',
                'spacing_consistent', 'segmentation_valid', 'has_nan_or_inf']
df_val[summary_cols].apply(pd.Series.value_counts).fillna(0).astype(int)

,files_complete,nifti_readable,shape_consistent,spacing_consistent,segmentation_valid,has_nan_or_inf
False,0,0,0,0,0,1251
True,1251,1251,1251,1251,1251,0


## 10. Final Summary

In [10]:
total          = len(df_val)
complete       = df_val['files_complete'].sum()
readable       = df_val['nifti_readable'].sum()
unreadable     = total - readable
shape_ok       = df_val['shape_consistent'].sum()
shape_bad      = total - shape_ok
spacing_ok     = df_val['spacing_consistent'].sum()
spacing_bad    = total - spacing_ok
seg_valid      = df_val['segmentation_valid'].sum()
seg_invalid    = total - seg_valid
nan_inf        = df_val['has_nan_or_inf'].sum()

print('=== VALIDATION SUMMARY ===')
print(f'Total cases               : {total}')
print(f'Complete cases (all files) : {complete}')
print(f'Unreadable NIfTI files     : {unreadable} cases')
print(f'Shape mismatch             : {shape_bad} cases')
print(f'Spacing mismatch           : {spacing_bad} cases')
print(f'Unexpected seg labels      : {seg_invalid} cases')
print(f'NaN/Inf in MRI volumes     : {nan_inf} cases')
print()

all_pass = (complete == total and unreadable == 0 and shape_bad == 0 and spacing_bad == 0
            and seg_invalid == 0 and nan_inf == 0)

if all_pass:
    print('All validation checks passed. The dataset is ready for preprocessing.')
else:
    print('Some validation checks failed. Review the issues above before proceeding.')

=== VALIDATION SUMMARY ===
Total cases               : 1251
Complete cases (all files) : 1251
Unreadable NIfTI files     : 0 cases
Shape mismatch             : 0 cases
Spacing mismatch           : 0 cases
Unexpected seg labels      : 0 cases
NaN/Inf in MRI volumes     : 0 cases

All validation checks passed. The dataset is ready for preprocessing.
